In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt

In [2]:
# глобальные параметры
K_NEIGHBORS = 20  # Число соседей для KNN
TOP_X = 10  # Сколько рекомендаций выводить
TEST_SIZE = 0.2  # Процент данных для теста
SAMPLE_ID = 1    # ID фильма, для которого составляем РС


In [3]:
# загрузка данных
def load_movielens_data():
    # Загружаем рейтинги (UserID::MovieID::Rating::Timestamp)
    ratings = pd.read_csv('ratings.dat', sep='::', engine='python',
                          names=['userId', 'movieId', 'rating', 'timestamp'],
                          encoding='ISO-8859-1')

    # Загружаем фильмы (MovieID::Title::Genres)
    movies = pd.read_csv('movies.dat', sep='::', engine='python',
                         names=['movieId', 'title', 'genres'],
                         encoding='ISO-8859-1')
    
    # Загружаем пользователей (UserID::Gender::Age::Occupation::Zip-code)
    users = pd.read_csv('users.dat', sep='::', engine='python',
                         names=['userId', 'gender', 'age', 'occupation', 'zip-code'],
                         encoding='ISO-8859-1')

    return ratings, movies

In [4]:
# создание разреженной матрицы (строки = фильмы, столбцы = пользователи, значения = оценки)
def create_sparse_matrix(df):
    pivot_table = df.pivot(index='movieId', columns='userId', values='rating').fillna(0)
    sparse_matrix = csr_matrix(pivot_table.values)
    return pivot_table, sparse_matrix

In [5]:
# функция рекомендаций
def get_recommendations(movie_id, n_recs=TOP_X):
    if movie_id not in train_pivot.index:
        return f"Фильм ID {movie_id} не найден в обучающей выборке."

    # Поиск индекса строки фильма в матрице
    movie_idx = train_pivot.index.get_loc(movie_id)

    # Находим ближайших соседей
    distances, indices = model_knn.kneighbors(
        train_pivot.iloc[movie_idx, :].values.reshape(1, -1),
        n_neighbors=n_recs + 1
    )

    recommendations = []
    for i in range(1, len(distances.flatten())):
        res_id = train_pivot.index[indices.flatten()[i]]
        recommendations.append({
            'title': movie_titles.get(res_id, "Unknown"),
            'distance': distances.flatten()[i],
            'id': res_id
        })
    return recommendations

In [6]:
# функция рассчета качества (RMSE)
def evaluate_model(test_df, train_pivot, model):
    actual_ratings = []
    predicted_ratings = []

    # Для оценки берем подмножество теста
    test_sample = test_df.sample(min(1000, len(test_df)), random_state=42)

    for _, row in test_sample.iterrows():
        uid, mid, actual_r = int(row['userId']), int(row['movieId']), row['rating']

        if mid in train_pivot.index and uid in train_pivot.columns:
            # ищем соседей фильма mid
            m_idx = train_pivot.index.get_loc(mid)
            distances, indices = model.kneighbors(
                train_pivot.iloc[m_idx, :].values.reshape(1, -1),
                n_neighbors=K_NEIGHBORS
            )

            # предсказание: средний рейтинг соседей, которые уже были оценены этим пользователем
            neighbor_ids = train_pivot.index[indices.flatten()[1:]]
            neighbor_ratings = []

            for nid in neighbor_ids:
                r = train_pivot.loc[nid, uid]
                if r > 0:  # если пользователь оценивал соседа
                    neighbor_ratings.append(r)

            if neighbor_ratings:
                predicted_ratings.append(np.mean(neighbor_ratings))
                actual_ratings.append(actual_r)

    if not actual_ratings:
        return None
    return sqrt(mean_squared_error(actual_ratings, predicted_ratings))

In [7]:
# загружаем данные из файлов
try:
    ratings_df, movies_df = load_movielens_data()
    print("Данные успешно загружены.")
except FileNotFoundError as e:
    print("Не удалось прочесть данные.")
    exit()

Данные успешно загружены.


In [8]:
# создаем словарь для быстрого поиска по ID
movie_titles = dict(zip(movies_df['movieId'], movies_df['title']))

In [9]:
# подготовка и моделирование

# Разделяем на train и test
train_data, test_data = train_test_split(ratings_df, test_size=TEST_SIZE, random_state=42)

train_pivot, train_sparse = create_sparse_matrix(train_data)

# Инициализация и обучение KNN
model_knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=K_NEIGHBORS, n_jobs=-1)
model_knn.fit(train_sparse)

,n_neighbors,20
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,-1


In [10]:
# вывод результатов
print(f"\nКонфигурация РС")
print(f"K-соседей: {K_NEIGHBORS} | Скрыто данных: {TEST_SIZE * 100}% | Топ рекомендаций: {TOP_X}")

sample_name = movie_titles.get(SAMPLE_ID, "Unknown")

print(f"\nРекомендации для фильма: '{sample_name}' (ID {SAMPLE_ID})")
recs = get_recommendations(SAMPLE_ID)

if isinstance(recs, list):
    for i, r in enumerate(recs, 1):
        print(f"{i}. {r['title']} (Distance: {r['distance']:.3f})")

rmse_val = evaluate_model(test_data, train_pivot, model_knn)
if rmse_val:
    print(f"\nКачество модели (RMSE): {rmse_val:.4f}")
else:
    print("\nНедостаточно пересечений в тестовой выборке для расчета RMSE.")



Конфигурация РС
K-соседей: 20 | Скрыто данных: 20.0% | Топ рекомендаций: 10

Рекомендации для фильма: 'Toy Story (1995)' (ID 1)
1. Toy Story 2 (1999) (Distance: 0.507)
2. Groundhog Day (1993) (Distance: 0.512)
3. Back to the Future (1985) (Distance: 0.538)
4. Aladdin (1992) (Distance: 0.538)
5. Bug's Life, A (1998) (Distance: 0.538)
6. Babe (1995) (Distance: 0.556)
7. Matrix, The (1999) (Distance: 0.561)
8. Princess Bride, The (1987) (Distance: 0.561)
9. Star Wars: Episode V - The Empire Strikes Back (1980) (Distance: 0.563)
10. Lion King, The (1994) (Distance: 0.570)

Качество модели (RMSE): 0.9567
